# E2SFCA Analysis for EmOC Accessibility
## Kano State, Nigeria

This notebook implements the Enhanced Two-Step Floating Catchment Area (E2SFCA) method to measure spatial accessibility to Emergency Obstetric Care (EmOC) facilities.

**Prerequisites**: Complete Notebook 01 (Data Preparation) first

This notebook:
1. Loads processed data from Notebook 01
2. Calculates distance matrices between demand and supply
3. Applies E2SFCA method for each facility category
4. Calculates accessibility scores
5. Classifies areas by deprivation level
6. Saves results for analysis in Notebook 03


> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate OD-Matrix on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point
from pathlib import Path
from shapely.geometry import Polygon
from shapely.geometry import box

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [2]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Kano/Data/raw/'
data_temp = '../Kano/Data/processed/'
model_outputs = '../Kano/Data/outputs/'

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

### 10min catchment area

In [3]:
# Function
# Gaussian decay: weight = exp(t² / beta), where beta = -d² / ln(W)
# - d: catchment threshold (travel time in seconds)
# - W: weight at the catchment boundary (i.e., when t = d, weight = W)
# - This ensures smooth decay from 1.0 (at t=0) to W=0.01 (at t=d),
#   meaning facilities beyond d still contribute but at <1% weight

from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [4]:
origin_dest = gpd.read_file('distances_duration_3_closest_Emoc.gpkg')
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        1    8.301005   12.122137        8.300491       12.121729   
1        1    8.301005   12.122137        8.300491       12.121729   
2        1    8.301005   12.122137        8.300491       12.121729   
3        2    8.319272   12.072376        8.318758       12.071968   
4        2    8.319272   12.072376        8.318758       12.071968   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        8.301519       12.122545   10.921890    10.0            110.0   
1        8.301519       12.122545   10.921890    10.0            110.0   
2        8.301519       12.122545   10.921890    10.0            110.0   
3        8.319786       12.072784   11.756603     1.0              8.0   
4        8.319786       12.072784   11.756603     1.0              8.0   

   pop_grid_pop  duration_seconds  distance_km  hcf_id  \
0    120.140793            374.30         5.77      14   
1    120.140793   

In [5]:
# 1. Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [6]:
# 2. Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [7]:
# 3. Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [8]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']
print(len(origin_dest_acc))
origin_dest_acc.head()

1672390


,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,374.30,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.166596,1.819541
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1679.81,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1708.62,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,470.08,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.059205,0.696052
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,1560.17,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.000000,0.000000


In [9]:
# 4. Sum the weighted population for each grid cell to get the total weighted population (Pop_W) for each grid cell.
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()
origin_dest_sum

,hcf_id,Pop_W
0,1,15603.539814
1,2,36512.813787
2,3,10659.897476
3,4,16032.091170
4,5,55939.302647
...,...,...
129,141,1666.569144
130,142,2708.502060
131,143,5616.732398
132,144,23939.736976


In [10]:
# 5. Merge the total weighted population back to the original DataFrame to have a complete dataset for analysis
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W_x,Pop_W_y
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.166596,1.819541,4406.686022
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000,10659.897476
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000,8555.357581
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.059205,0.696052,4406.686022
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.000000,0.000000,10659.897476


In [11]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [12]:
# The supply value is set based on the type of healthcare facility, with different weights assigned to public and private facilities.
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

### Sensitivity Analysis for Supply Map

In [15]:
# ============================================================
# SENSITIVITY ANALYSIS: Testing robustness of supply weights
# ============================================================
# Re-runs E2SFCA with different supply weight scenarios to check
# whether the final accessibility classification is sensitive to
# the choice of weights.

from scipy.stats import spearmanr

scenarios = {
    'Baseline':      {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.7,
                      'Public Basic EmOC': 0.5,  'Private Basic EmOC': 0.35},
    'Equal':         {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 1.0,
                      'Public Basic EmOC': 1.0,  'Private Basic EmOC': 1.0},
    'Public Only':   {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.3,
                      'Public Basic EmOC': 0.5,  'Private Basic EmOC': 0.15},
    'Wider Gap':     {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.5,
                      'Public Basic EmOC': 0.3,  'Private Basic EmOC': 0.15},
}

def run_e2sfca_with_supply(origin_dest_input, supply_map, d, beta):
    """Re-run E2SFCA steps with a given supply_map."""
    df = origin_dest_input.copy()
    
    df['Weight'] = np.exp(-df['duration_seconds']**2 / beta)
    df['Pop_W'] = df['population'] * df['Weight']
    
    pop_w_sum = df.groupby('hcf_id')['Pop_W'].sum().reset_index()
    pop_w_sum.columns = ['hcf_id', 'Pop_W_S']
    df = df.merge(pop_w_sum, on='hcf_id', how='left')
    
    df['supply'] = df['Local_Validation'].map(supply_map)
    df['supply_demand_ratio'] = df['supply'] / df['Pop_W_S']
    df['supply_W'] = df['supply_demand_ratio'] * df['Weight']
    df['Accessibility'] = df.groupby('grid_id')['supply_W'].transform('sum')
    df['Accessibility_standard'] = (df['Accessibility'] - df['Accessibility'].min()) / \
                                    (df['Accessibility'].max() - df['Accessibility'].min())
    
    result = df.drop_duplicates(subset='grid_id')[
        ['grid_id', 'Accessibility', 'Accessibility_standard']
    ].copy()
    return result


# Run all scenarios
results = {}
for name, smap in scenarios.items():
    results[name] = run_e2sfca_with_supply(origin_dest, smap, d, beta)
    print(f"✓ {name} complete")

# ---- Results ----
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS RESULTS")
print("=" * 60)

# 1. Raw accessibility values (before normalisation)
print("\nRaw Accessibility (before normalisation):")
print(f"  {'Scenario':15s}  {'Mean':>12s}  {'Median':>12s}  {'Std':>12s}")
for name, res in results.items():
    a = res['Accessibility']
    print(f"  {name:15s}  {a.mean():12.6f}  {a.median():12.6f}  {a.std():12.6f}")

# 2. Percentage change in raw values vs Baseline
baseline_raw_mean = results['Baseline']['Accessibility'].mean()
print("\nChange in raw mean vs Baseline:")
for name, res in results.items():
    if name == 'Baseline':
        continue
    change = (res['Accessibility'].mean() - baseline_raw_mean) / baseline_raw_mean * 100
    print(f"  {name:15s}: {change:+.1f}%")

# 3. Spearman correlation (normalised)
baseline = results['Baseline'].set_index('grid_id')['Accessibility_standard']
print("\nSpearman correlation with Baseline (normalised):")
for name, res in results.items():
    if name == 'Baseline':
        continue
    other = res.set_index('grid_id')['Accessibility_standard']
    common = baseline.index.intersection(other.index)
    corr, pval = spearmanr(baseline[common], other[common])
    print(f"  {name:15s}: r = {corr:.4f}  (p = {pval:.2e})")

# 4. Classification agreement
x_threshold = 0.005
y_threshold = 0.020

print(f"\nClassification agreement (x={x_threshold}, y={y_threshold}):")
baseline_class = np.where(baseline >= y_threshold, 0, np.where(baseline >= x_threshold, 1, 2))

for name, res in results.items():
    if name == 'Baseline':
        continue
    other = res.set_index('grid_id')['Accessibility_standard']
    common = baseline.index.intersection(other.index)
    other_class = np.where(other[common] >= y_threshold, 0, 
                  np.where(other[common] >= x_threshold, 1, 2))
    agreement = np.mean(baseline_class[baseline.index.isin(common)] == other_class) * 100
    print(f"  {name:15s}: {agreement:.1f}% grids same category")

✓ Baseline complete
✓ Equal complete
✓ Public Only complete
✓ Wider Gap complete

SENSITIVITY ANALYSIS RESULTS

Raw Accessibility (before normalisation):
  Scenario                 Mean        Median           Std
  Baseline             0.000049      0.000001      0.000175
  Equal                0.000065      0.000002      0.000245
  Public Only          0.000028      0.000001      0.000087
  Wider Gap            0.000038      0.000001      0.000130

Change in raw mean vs Baseline:
  Equal          : +34.7%
  Public Only    : -42.3%
  Wider Gap      : -21.8%

Spearman correlation with Baseline (normalised):
  Equal          : r = 0.9994  (p = 0.00e+00)
  Public Only    : r = 0.9977  (p = 0.00e+00)
  Wider Gap      : r = 0.9996  (p = 0.00e+00)

Classification agreement (x=0.005, y=0.02):
  Equal          : 98.1% grids same category
  Public Only    : 94.6% grids same category
  Wider Gap      : 97.9% grids same category


In [21]:
# 2. CATCHMENT TIME SENSITIVITY
# ============================================================
print("\n" + "=" * 60)
print("2. CATCHMENT TIME SENSITIVITY")
print("=" * 60)

catchment_scenarios = {'10min': 10*60, '15min': 15*60, '20min': 20*60, '30min': 30*60}

print("\nAccessibility distribution per catchment:")
for name, d_val in catchment_scenarios.items():
    beta_val = -d_val**2 / log(W)
    res = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], d_val, beta_val)
    median_val = res['Accessibility'].median()
    non_tiny = (res['Accessibility_standard'] > 0.001).sum()
    print(f"  {name}: median={median_val:.8f}, grids with meaningful access={non_tiny}/{len(res)}")


2. CATCHMENT TIME SENSITIVITY

Accessibility distribution per catchment:
  10min: median=0.00000126, grids with meaningful access=61486/167239
  15min: median=0.00001633, grids with meaningful access=95496/167239
  20min: median=0.00003714, grids with meaningful access=123842/167239
  30min: median=0.00005874, grids with meaningful access=154096/167239


In [ ]:
# 3. CLASSIFICATION AGREEMENT VS 10MIN BASELINE
# ============================================================
print("\n" + "=" * 60)
print("3. CLASSIFICATION AGREEMENT VS 10MIN BASELINE")
print("=" * 60)

beta_10 = -(10*60)**2 / log(W)
baseline_10 = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], 10*60, beta_10)
base = baseline_10.set_index('grid_id')['Accessibility_standard']
base_class = np.where(base >= y_threshold, 0, np.where(base >= x_threshold, 1, 2))

for name, d_val in {'15min': 15*60, '20min': 20*60, '30min': 30*60}.items():
    beta_val = -d_val**2 / log(W)
    res = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], d_val, beta_val)
    other = res.set_index('grid_id')['Accessibility_standard']
    common = base.index.intersection(other.index)
    other_class = np.where(other[common] >= y_threshold, 0, np.where(other[common] >= x_threshold, 1, 2))
    agreement = np.mean(base_class[base.index.isin(common)] == other_class) * 100
    corr, _ = spearmanr(base[common], other[common])
    print(f"  {name}: {agreement:.1f}% same category, Spearman r={corr:.4f}")


3. CLASSIFICATION AGREEMENT VS 10MIN BASELINE
  15min: 87.1% same category, Spearman r=0.9900
  20min: 73.9% same category, Spearman r=0.9621
  30min: 55.8% same category, Spearman r=0.8636

✓ All sensitivity analyses complete


In [59]:
# Calculate the supply-demand ratio by dividing the supply value by the total weighted population (Pop_W_S) for each grid cell. This ratio provides an indication of the accessibility of healthcare facilities relative to the demand from the population in each grid cell.
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'] = (
    origin_dest_acc['supply_demand_ratio']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [60]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [61]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [62]:
# Normalize the Accessibility Index using Min-Max Scaling to bring the values between 0 and 1, making it easier to interpret and compare across different grid cells.
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.107341,Public Comprehensive EmOC,0.166596,1.819541,4406.686022,1.0,0.000227,0.000038,0.000038,0.004124
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.056152,Public Comprehensive EmOC,0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000038,0.004124
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.058087,Public Comprehensive EmOC,0.000000,0.000000,8555.357581,1.0,0.000117,0.000000,0.000038,0.004124
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,12.107341,Public Comprehensive EmOC,0.059205,0.696052,4406.686022,1.0,0.000227,0.000013,0.000013,0.001466
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,12.056152,Public Comprehensive EmOC,0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000013,0.001466


In [63]:
# Check the maximum value of the standardized Accessibility Index to ensure it is correctly normalized to 1.
max(origin_dest_acc.Accessibility_standard)

1.0

In [64]:
# Convert the final DataFrame to a GeoDataFrame and save it as a GeoPackage
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'accessibility-index.gpkg'
gdf.to_file(gpkg_path, layer="accessibility-index", driver="GPKG")

### 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [79]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'accessibility-index.gpkg')

# Select columns to keep and reorder them
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 
                             'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'hcf_id', 
                             'facility_name', 'Local_Validation', 'duration_seconds', 'distance_km', 
                             'Accessibility_standard', 'geometry']]

### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [80]:
# Classify the accessibility into three categories based on the standardized Accessibility Index: 0 for High accessibility, 1 for Medium accessibility, and 2 for Low accessibility. The thresholds for classification are set at 0.005 and 0.02, which can be adjusted based on the distribution of the Accessibility Index in the dataset.
results_grid['result'] = 2  # Initialize all cells to 2 (Low accessibility)
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [81]:
results_grid.to_file(data_temp + 'accessibility-index-classified.gpkg', 
                     layer='accessibility-index-classified', driver='GPKG')

In [82]:
# The E2SFCA accessibility index is calculated based on the 3 closest healthcare 
# facilities for each grid cell. Since all 3 rows per grid cell share the same 
# accessibility value, we select the row with the minimum duration_seconds for each grid_id as representative to avoid duplication.
idx = results_grid.groupby('grid_id')['duration_seconds'].idxmin()
results_grid_dedup = results_grid.loc[idx].reset_index(drop=True)
print(len(results_grid_dedup))
results_grid_dedup.head()

print(results_grid_dedup['Local_Validation'].value_counts())

167239
Local_Validation
Private Comprehensive EmOC    120825
Public Comprehensive EmOC      36954
Private Basic EmOC              9203
Public Basic EmOC                257
Name: count, dtype: int64


In [83]:
category_counts = results_grid_dedup['result'].value_counts()
print(category_counts)

result
2    130081
1     24886
0     12272
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [70]:
# Identify focus areas for intervention based on the standardized Accessibility Index.
results_grid_dedup['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.000001) & (results_grid_dedup['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.003) & (results_grid_dedup['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.019) & (results_grid_dedup['Accessibility_standard'] < 0.03), 'focused'] = 1

In [71]:
category_counts = results_grid_dedup['focused'].value_counts()
print(category_counts)

focused
0    147064
1     20175
Name: count, dtype: int64


In [72]:
# Rename columns for clarity and consistency
results_grid_dedup = results_grid_dedup.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max',
    'Accessibility_standard': 'Accessibility_Index_Standard'
})

results_grid_dedup.head()

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,hcf_id,facility_name,Local_Validation,duration_seconds,distance_km,Accessibility_Index_Standard,geometry,result,focused
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",2,1
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",2,0
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ...",0,1
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ...",0,1
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ...",2,0


In [74]:
# Dataset 2: 
dataset_2 = results_grid_dedup.copy()
dataset_2 = gpd.GeoDataFrame(
    results_grid_dedup,
    geometry='geometry',
    crs='EPSG:4326'
)

dataset_2 = dataset_2[['grid_id', 'longitude', 'latitude', 'lon_min', 'lon_max', 'lat_min', 'lat_max', 'Accessibility_Index_Standard', 'result', 
                         'focused', 'geometry']]
output_gpkg_path = model_outputs + 'deprivation-classification.gpkg'
dataset_2.to_file(output_gpkg_path, layer='deprivation-classification', driver='GPKG')
print(f"\n✓ Saved dataset_2 with {len(dataset_2)} records to {output_gpkg_path}")
dataset_2.head()


✓ Saved dataset_2 with 167239 records to ../Kano/Data/outputs/deprivation-classification.gpkg


,grid_id,longitude,latitude,lon_min,lon_max,lat_min,lat_max,Accessibility_Index_Standard,result,focused,geometry
0,1,8.301005,12.122137,8.300491,8.301519,12.121729,12.122545,0.004124,2,1,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ..."
1,2,8.319272,12.072376,8.318758,8.319786,12.071968,12.072784,0.001466,2,0,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ..."
2,3,8.330126,12.110716,8.329612,8.330640,12.110308,12.111124,0.021559,0,1,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ..."
3,4,8.330079,12.108269,8.329565,8.330593,12.107861,12.108676,0.024239,0,1,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ..."
4,5,8.332575,12.027513,8.332061,8.333088,12.027105,12.027921,0.000000,2,0,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ..."


In [ ]:
dataset_2 = gpd.read_file(model_outputs + 'deprivation-classification.gpkg')
dataset_2.to_csv(model_outputs + 'deprivation-classification.csv', index=False)

## E2SFCA Catchment Area Optimisation
#### Comparing 10 / 15 / 20-Minute Catchment Areas Against Validation Data
**Objective**: Test how different maximum catchment travel times affect the E2SFCA accessibility index, automatically find the best classification thresholds for each scenario, and compare all results against community validation data.

In [15]:
# Load OD matrix and validation
origin_dest = gpd.read_file('distances_duration_3_closest_Emoc.gpkg')
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest = origin_dest.dropna(subset=['duration_seconds'])

validation = pd.read_csv('2025-08-01T09_35_49+00_00_o6ix.csv')

In [23]:
# Select pilot city
pilot = 'Kano'  # Change this to the desired pilot city
# Load PAR data and filter by pilot city
# par_expanded = pd.read_csv('/content/expanded_data.csv') # You need to upload this file
par_expanded = validation[validation['output_model_city_name'] == f'{pilot.capitalize()}']

In [24]:
# Describe the shape of the validation data
print("Shape of PAR Data", par_expanded.shape)
par_expanded['output_model_subdomain_name'].value_counts()

Shape of PAR Data (22961, 16)


output_model_subdomain_name
Morphological Informality                      10914
Emergency Obstetric Care Access Deprivation     4434
General Healthcare Access Deprivation           4032
Road Access Deprivation                         3581
Name: count, dtype: int64

In [26]:
# Enter subdomain
subdomain = 'Emergency Obstetric Care Access Deprivation'  # Change this to the desired subdomain

# Filter Data by Subdomain
par_subdomain = par_expanded[par_expanded['output_model_subdomain_name'] == subdomain]
par_subdomain

,id,created_at,validation,user_id,user_background,user_map_usage,output_id,output_model,output_model_city_name,output_model_city_country,output_model_subdomain_name,output_result,output_latitude,output_longitude,lat_round,lon_round
141,3e2c4ed2-5d70-4366-87cb-31b2fc708ad7,2025-05-22T14:30:43.470779+00:00,1.0,e039d820-cd6e-4360-b92b-8729c3354bd4,['Research'],3.0,594583b8-21ac-40a7-8bad-47d5d4853571,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.040564,8.572567,12.041,8.573
197,b9d75a79-f95b-474a-b635-ae5dabc701d4,2025-05-22T14:31:08.713542+00:00,1.0,e039d820-cd6e-4360-b92b-8729c3354bd4,['Research'],3.0,f699230f-14d5-4309-a765-632849cbe817,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.041379,8.573595,12.041,8.574
262,46c1fe60-2017-418a-9c8e-58fc9d3f0313,2025-05-22T14:31:12.345971+00:00,1.0,e039d820-cd6e-4360-b92b-8729c3354bd4,['Research'],3.0,70f2736e-12a6-4eb2-9485-8538bdeade0f,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.039748,8.574574,12.040,8.575
263,af24f34b-f8cf-4d1b-94fb-0aa451766abe,2025-05-22T14:31:17.362707+00:00,0.0,11072e21-e8b7-45e6-88ff-b1569e233941,['Community Member'],3.0,59de5b75-eb3e-4c14-b789-57bae0f34653,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,0,12.034854,8.563351,12.035,8.563
325,64490bd3-c8bf-4748-9e8a-249d070c7397,2025-05-20T11:51:06.297704+00:00,1.0,3617df69-2dae-48f8-b1dc-7f4809b0d7ad,['Research'],2.0,c9d663ea-b9d9-461b-ab7a-67293f6c20f7,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,11.995702,8.521111,11.996,8.521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63154,dae2fbea-cc3b-4ffe-91ff-21526daf5742,2025-07-02T18:27:20.657966+00:00,1.0,9bd770ff-83bb-48d6-823f-04ceb2257a8c,NaN,2.0,6caacc54-94e6-4dd3-8e34-bad74e461ea0,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.069113,8.473990,12.069,8.474
63155,80a1c4a4-ade4-4b41-afbf-d47c960693ba,2025-07-02T18:27:22.47451+00:00,1.0,9bd770ff-83bb-48d6-823f-04ceb2257a8c,NaN,2.0,4ffe90eb-d58d-4cb8-a10e-6e1c8c42d3b1,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.069113,8.472979,12.069,8.473
63156,04868cec-be51-4293-b659-c3773f28db55,2025-07-02T18:27:25.818082+00:00,1.0,9bd770ff-83bb-48d6-823f-04ceb2257a8c,NaN,2.0,974267d3-787d-4c4b-8fc4-89dd61d3f3ea,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.069113,8.471967,12.069,8.472
63158,b31a8744-b9c3-423a-a5c2-c7b28d3c82d4,2025-07-02T18:27:36.099856+00:00,0.0,9bd770ff-83bb-48d6-823f-04ceb2257a8c,NaN,2.0,e2fb1c54-afea-4ef6-aefc-25c2cc6ae5d1,4c44b4ef-407a-469c-af5d-185aa83c7133,Kano,Nigeria,Emergency Obstetric Care Access Deprivation,1,12.068297,8.471951,12.068,8.472


In [27]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        1    8.301005   12.122137        8.300491       12.121729   
1        1    8.301005   12.122137        8.300491       12.121729   
2        1    8.301005   12.122137        8.300491       12.121729   
3        2    8.319272   12.072376        8.318758       12.071968   
4        2    8.319272   12.072376        8.318758       12.071968   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        8.301519       12.122545   10.921890    10.0            110.0   
1        8.301519       12.122545   10.921890    10.0            110.0   
2        8.301519       12.122545   10.921890    10.0            110.0   
3        8.319786       12.072784   11.756603     1.0              8.0   
4        8.319786       12.072784   11.756603     1.0              8.0   

   pop_grid_pop  duration_seconds  distance_km  hcf_id  \
0    120.140793            374.30         5.77      14   
1    120.140793   

## 15mins

In [40]:
# Function
# Gaussian decay: weight = exp(t² / beta), where beta = -d² / ln(W)
# - d: catchment threshold (travel time in seconds)
# - W: weight at the catchment boundary (i.e., when t = d, weight = W)
# - This ensures smooth decay from 1.0 (at t=0) to W=0.01 (at t=d),
#   meaning facilities beyond d still contribute but at <1% weight

from math import *
d = 15 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

175889.265170817


### E2SFCA Pipeline

In [41]:
df = origin_dest.copy()

supply_map = {
    'Public Comprehensive EmOC': 1.0,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35,
}

# Gaussian decay weight
df['Weight'] = np.exp(-df['duration_seconds']**2 / beta)

# Step A: weighted population → demand at each facility
df['Pop_W'] = df['population'] * df['Weight']
demand = df.groupby('hcf_id')['Pop_W'].sum().reset_index(name='Demand')
df = df.merge(demand, on='hcf_id')

# Step B: supply / demand ratio × weight → accessibility
df['supply'] = df['Local_Validation'].map(supply_map)
df['R'] = (df['supply'] / df['Demand']).replace([np.inf, -np.inf], 0).fillna(0)
df['R_weighted'] = df['R'] * df['Weight']
df['Accessibility'] = df.groupby('grid_id')['R_weighted'].transform('sum')

# Normalise to [0, 1]
a_min, a_max = df['Accessibility'].min(), df['Accessibility'].max()
df['Accessibility_standard'] = (df['Accessibility'] - a_min) / (a_max - a_min)

# De-duplicate: one row per grid cell (keep shortest travel time)
idx = df.groupby('grid_id')['duration_seconds'].idxmin()
result_15mins = df.loc[idx].reset_index(drop=True)

### Join Validation Points to Grid Cells
Each validation record has `output_latitude` / `output_longitude`. 

In [42]:
# Merge validation results to accessibility index by rounded coordinates
merged = result_15mins.merge(
    par_subdomain[['output_latitude', 'output_longitude', 'output_result', 'validation']],
    left_on  = [result_15mins['origin_lat'].round(7), result_15mins['origin_lon'].round(7)],
    right_on = [par_subdomain['output_latitude'].round(7), par_subdomain['output_longitude'].round(7)],
    how = 'inner'
).drop(columns=['key_0', 'key_1', 'output_latitude', 'output_longitude'])

print(f"✓ Matched {len(merged):,} rows")

# Majority vote per grid cell
matched = (merged.groupby('grid_id')
    .agg(
        validation=('validation', lambda x: x.mode().iloc[0]),
        vote_count=('validation', 'count'),
        agreement=('validation', lambda x: (x == x.mode().iloc[0]).mean()),
        output_result=('output_result', 'first'),
        Accessibility_standard=('Accessibility_standard', 'first'),
    )
    .reset_index()
)

print(f"✓ {len(merged):,} votes → {len(matched):,} grid cells (majority vote)")
print(f"  Mean votes per cell: {matched['vote_count'].mean():.1f}")
print(f"  Mean agreement:      {matched['agreement'].mean():.1%}")
print(f"\nValidation distribution (after majority vote):")
print(matched['validation'].value_counts().sort_index())

✓ Matched 4,434 rows
✓ 4,434 votes → 4,245 grid cells (majority vote)
  Mean votes per cell: 1.0
  Mean agreement:      99.2%

Validation distribution (after majority vote):
validation
0.0     906
1.0    1408
2.0    1931
Name: count, dtype: int64


### Automatic Threshold Optimisation

Instead of manually setting the two classification thresholds, we perform a **grid search** over all plausible `(t_low, t_high)` pairs and pick the combination that maximises the **weighted F1-score** against the validation data.

Classification rule:
- `Accessibility_standard > t_high` → **0** (Low deprivation)
- `Accessibility_standard > t_low`  → **1** (Medium deprivation)  
- otherwise → **2** (High deprivation)

In [43]:
from sklearn.metrics import f1_score

acc_values = matched['Accessibility_standard'].values
y_true = matched['validation'].values

# Generate candidate thresholds from the data (percentile range)
p_min, p_max = np.percentile(acc_values[acc_values > 0], [1, 99])
candidates = np.linspace(p_min, p_max, 300)

best_f1 = -1
best_t_low = 0
best_t_high = 0

for i, t_low in enumerate(candidates):
    for t_high in candidates[i+1:]:
        # Classify
        y_pred = np.where(acc_values > t_high, 0, np.where(acc_values > t_low, 1, 2))
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t_low = t_low
            best_t_high = t_high

print(f"✓ Best thresholds found:")
print(f"  t_low  = {best_t_low:.6f}   (boundary between High ↔ Medium)")
print(f"  t_high = {best_t_high:.6f}   (boundary between Medium ↔ Low)")
print(f"  Weighted F1 = {best_f1:.4f}")

✓ Best thresholds found:
  t_low  = 0.008760   (boundary between High ↔ Medium)
  t_high = 0.018916   (boundary between Medium ↔ Low)
  Weighted F1 = 0.6364


In [44]:
from sklearn.metrics import classification_report
class_names = ['Low deprivation (0)', 'Medium deprivation (1)', 'High deprivation (2)']

print("=== Original Model (output_result) vs Validation ===")
print(classification_report(
    matched['validation'], 
    matched['output_result'], 
    target_names=class_names, digits=2
))

=== Original Model (output_result) vs Validation ===
                        precision    recall  f1-score   support

   Low deprivation (0)       0.67      0.53      0.59       906
Medium deprivation (1)       0.71      0.69      0.70      1408
  High deprivation (2)       0.74      0.83      0.78      1931

              accuracy                           0.72      4245
             macro avg       0.71      0.68      0.69      4245
          weighted avg       0.72      0.72      0.71      4245



In [45]:
y_pred_15 = np.where(acc_values > best_t_high, 0, np.where(acc_values > best_t_low, 1, 2))

print("=== 15-Minute Catchment (Optimised Thresholds) vs Validation ===")
print(f"    t_low = {best_t_low:.6f},  t_high = {best_t_high:.6f}")
print()
print(classification_report(
    matched['validation'], 
    y_pred_15, 
    target_names=class_names, digits=2
))

=== 15-Minute Catchment (Optimised Thresholds) vs Validation ===
    t_low = 0.008760,  t_high = 0.018916

                        precision    recall  f1-score   support

   Low deprivation (0)       0.57      0.51      0.54       906
Medium deprivation (1)       0.61      0.51      0.56      1408
  High deprivation (2)       0.69      0.80      0.74      1931

              accuracy                           0.64      4245
             macro avg       0.62      0.61      0.61      4245
          weighted avg       0.64      0.64      0.64      4245



## 20mins

In [47]:
# Function
# Gaussian decay: weight = exp(t² / beta), where beta = -d² / ln(W)
# - d: catchment threshold (travel time in seconds)
# - W: weight at the catchment boundary (i.e., when t = d, weight = W)
# - This ensures smooth decay from 1.0 (at t=0) to W=0.01 (at t=d),
#   meaning facilities beyond d still contribute but at <1% weight

from math import *
d = 20 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

312692.02697034134


In [49]:
df = origin_dest.copy()

supply_map = {
    'Public Comprehensive EmOC': 1.0,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35,
}

# Gaussian decay weight
df['Weight'] = np.exp(-df['duration_seconds']**2 / beta)

# Step A: weighted population → demand at each facility
df['Pop_W'] = df['population'] * df['Weight']
demand = df.groupby('hcf_id')['Pop_W'].sum().reset_index(name='Demand')
df = df.merge(demand, on='hcf_id')

# Step B: supply / demand ratio × weight → accessibility
df['supply'] = df['Local_Validation'].map(supply_map)
df['R'] = (df['supply'] / df['Demand']).replace([np.inf, -np.inf], 0).fillna(0)
df['R_weighted'] = df['R'] * df['Weight']
df['Accessibility'] = df.groupby('grid_id')['R_weighted'].transform('sum')

# Normalise to [0, 1]
a_min, a_max = df['Accessibility'].min(), df['Accessibility'].max()
df['Accessibility_standard'] = (df['Accessibility'] - a_min) / (a_max - a_min)

# De-duplicate: one row per grid cell (keep shortest travel time)
idx = df.groupby('grid_id')['duration_seconds'].idxmin()
result_20mins = df.loc[idx].reset_index(drop=True)

In [51]:
# Merge validation results to accessibility index by rounded coordinates
merged_20mins = result_20mins.merge(
    par_subdomain[['output_latitude', 'output_longitude', 'output_result', 'validation']],
    left_on  = [result_20mins['origin_lat'].round(7), result_20mins['origin_lon'].round(7)],
    right_on = [par_subdomain['output_latitude'].round(7), par_subdomain['output_longitude'].round(7)],
    how = 'inner'
).drop(columns=['key_0', 'key_1', 'output_latitude', 'output_longitude'])

print(f"✓ Matched {len(merged_20mins):,} rows")

# Majority vote per grid cell
matched_20mins = (merged_20mins.groupby('grid_id')
    .agg(
        validation=('validation', lambda x: x.mode().iloc[0]),
        vote_count=('validation', 'count'),
        agreement=('validation', lambda x: (x == x.mode().iloc[0]).mean()),
        output_result=('output_result', 'first'),
        Accessibility_standard=('Accessibility_standard', 'first'),
    )
    .reset_index()
)

print(f"✓ {len(merged_20mins):,} votes → {len(matched_20mins):,} grid cells (majority vote)")
print(f"  Mean votes per cell: {matched_20mins['vote_count'].mean():.1f}")
print(f"  Mean agreement:      {matched_20mins['agreement'].mean():.1%}")
print(f"\nValidation distribution (after majority vote):")
print(matched_20mins['validation'].value_counts().sort_index())

✓ Matched 4,434 rows
✓ 4,434 votes → 4,245 grid cells (majority vote)
  Mean votes per cell: 1.0
  Mean agreement:      99.2%

Validation distribution (after majority vote):
validation
0.0     906
1.0    1408
2.0    1931
Name: count, dtype: int64


### Automatic Threshold Optimisation

Instead of manually setting the two classification thresholds, we perform a **grid search** over all plausible `(t_low, t_high)` pairs and pick the combination that maximises the **weighted F1-score** against the validation data.

Classification rule:
- `Accessibility_standard > t_high` → **0** (Low deprivation)
- `Accessibility_standard > t_low`  → **1** (Medium deprivation)  
- otherwise → **2** (High deprivation)

In [53]:
from sklearn.metrics import f1_score

acc_values = matched_20mins['Accessibility_standard'].values
y_true = matched_20mins['validation'].values

# Generate candidate thresholds from the data (percentile range)
p_min, p_max = np.percentile(acc_values[acc_values > 0], [1, 99])
candidates = np.linspace(p_min, p_max, 300)

best_f1 = -1
best_t_low = 0
best_t_high = 0

for i, t_low in enumerate(candidates):
    for t_high in candidates[i+1:]:
        # Classify
        y_pred = np.where(acc_values > t_high, 0, np.where(acc_values > t_low, 1, 2))
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t_low = t_low
            best_t_high = t_high

print(f"✓ Best thresholds found:")
print(f"  t_low  = {best_t_low:.6f}   (boundary between High ↔ Medium)")
print(f"  t_high = {best_t_high:.6f}   (boundary between Medium ↔ Low)")
print(f"  Weighted F1 = {best_f1:.4f}")

✓ Best thresholds found:
  t_low  = 0.010007   (boundary between High ↔ Medium)
  t_high = 0.015801   (boundary between Medium ↔ Low)
  Weighted F1 = 0.5897


In [55]:
from sklearn.metrics import classification_report
class_names = ['Low deprivation (0)', 'Medium deprivation (1)', 'High deprivation (2)']

print("=== Original Model (output_result) vs Validation ===")
print(classification_report(
    matched['validation'], 
    matched['output_result'], 
    target_names=class_names, digits=2
))

=== Original Model (output_result) vs Validation ===
                        precision    recall  f1-score   support

   Low deprivation (0)       0.67      0.53      0.59       906
Medium deprivation (1)       0.71      0.69      0.70      1408
  High deprivation (2)       0.74      0.83      0.78      1931

              accuracy                           0.72      4245
             macro avg       0.71      0.68      0.69      4245
          weighted avg       0.72      0.72      0.71      4245



In [54]:
y_pred_20 = np.where(acc_values > best_t_high, 0, np.where(acc_values > best_t_low, 1, 2))

print("=== 20-Minute Catchment (Optimised Thresholds) vs Validation ===")
print(f"    t_low = {best_t_low:.6f},  t_high = {best_t_high:.6f}")
print()
print(classification_report(
    matched_20mins['validation'], 
    y_pred_20, 
    target_names=class_names, digits=2
))

=== 20-Minute Catchment (Optimised Thresholds) vs Validation ===
    t_low = 0.010007,  t_high = 0.015801

                        precision    recall  f1-score   support

   Low deprivation (0)       0.44      0.56      0.49       906
Medium deprivation (1)       0.59      0.42      0.49      1408
  High deprivation (2)       0.68      0.74      0.71      1931

              accuracy                           0.59      4245
             macro avg       0.57      0.57      0.56      4245
          weighted avg       0.60      0.59      0.59      4245

